In [0]:
# === Init cell with config (RUN FIRST) ===
from pyspark.sql.functions import col, count, avg, max, min, sum, round, concat_ws, lit, when

storage_account_hierarchical = "foremstorageaccount"
silver_adsl_path = f"abfss://silver@{storage_account_hierarchical}.dfs.core.windows.net/"
gold_adsl_path = f"abfss://gold@{storage_account_hierarchical}.dfs.core.windows.net/"

# spark.read.format("delta") - create a DataFrameReader object and configure it to expect Delta Lake format data
# load - read the data from the specified path and return a DataFrame
df_silver = spark.read.format("delta").load(silver_adsl_path)

# filter articles published in 11.2022 and in 02.2026 (partial data)
df_silver = df_silver.filter((col("published_at").cast("date") >= "2022-12-01") & (col("published_at").cast("date") < "2026-02-01"))



In [0]:
from pyspark.sql.functions import col, when

conditionComments = ((col("comments_count") > 0) & (col("comments_count").isNotNull()))
conditonPublicReactions = ((col("public_reactions_count") > 0) & (col("public_reactions_count").isNotNull()))

# Articles with comments OR reactions (either one or both)
df_gold_articles_fact_table = (
    df_silver
    .withColumn("has_comments", when(conditionComments, 1).otherwise(0))
    .withColumn("has_reactions", when(conditonPublicReactions, 1).otherwise(0))
    .select(
        "id",
        "published_at",
        "comments_count",
        "public_reactions_count",
        "has_comments",
        "has_reactions",
        "language",
        "user_id",
        "reading_time_minutes",
        "year",
        "month"
    )
)
# df_gold_articles_fact_table.limit(10).display()

df_gold_articles_fact_table.write.format("delta") \
    .partitionBy("year", "month") \
    .mode("overwrite") \
    .save(f"{gold_adsl_path}articles_fact")

In [0]:
from pyspark.sql.functions import explode, col, count, row_number, desc

df_tags_fact_table = df_silver.select(
    "id",
    "reading_time_minutes",
    "comments_count",
    "public_reactions_count",
    "user_id",
    "date",
    "year",
    "month",
    explode(col("tag_list")).alias("tag"),
).filter(col("tag").isNotNull() & (col("tag") != "")) # filter articles with no tags

df_tags_fact_table.write.format("delta") \
    .mode("overwrite") \
    .save(f"{gold_adsl_path}tags_fact")


In [0]:
from pyspark.sql.functions import col, explode, count, desc
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# 1️⃣ Explode tags once
df_tags = (
    df_silver
    .select("year", "month", explode(col("tag_list")).alias("tag"))
    .filter(col("tag").isNotNull() & (col("tag") != ""))
)

# 2️⃣ Articles per tag per month
df_tag_month_2 = (
    df_tags
    .groupBy("year", "month", "tag")
    .agg(count("*").alias("articles_count"))
)

# 3️⃣ Total articles per month (from non-exploded table)
df_total_month_2 = (
    df_silver
    .groupBy("year", "month")
    .agg(count("*").alias("total_articles_month"))
)

# 4️⃣ Find Top 10 tags PER MONTH
window_month = Window.partitionBy("year", "month").orderBy(desc("articles_count"))

df_top10_each_month = (
    df_tag_month_2
    .withColumn("rank", row_number().over(window_month))
    .filter(col("rank") <= 100)
    .drop("rank")
)

# 5️⃣ Create UNION list of tags that appeared in any monthly Top 10
top_tags_union = (
    df_top10_each_month
    .select("tag")
    .distinct()
)

# 6️⃣ Keep only those tags across ALL months
df_tag_filtered = (
    df_tag_month_2
    .join(top_tags_union, on="tag", how="inner")
)

# 7️⃣ Join totals and compute percentage
df_final = (
    df_tag_filtered
    .join(df_total_month_2, ["year", "month"], "inner")
    .withColumn(
        "percentage",
        (col("articles_count") / col("total_articles_month")) * 100
    )
    .orderBy("year", "month", desc("articles_count"))
)

# df_final.display()

df_final.write.format("delta") \
    .mode("overwrite") \
    .save(f"{gold_adsl_path}tags_per_month_fact")

In [0]:
from pyspark.sql.functions import col, when

conditionComments = ((col("comments_count") > 0) & (col("comments_count").isNotNull()))
conditonPublicReactions = ((col("public_reactions_count") > 0) & (col("public_reactions_count").isNotNull()))

# Articles with comments OR reactions (either one or both)
df_gold_full_articles_fact_table = (
    df_silver
    .withColumn("has_comments", when(conditionComments, 1).otherwise(0))
    .withColumn("has_reactions", when(conditonPublicReactions, 1).otherwise(0))
    .select(
        "id",
        "title",
        "created_at",
        "published_at",
        "user",
        "tag_list",
        "positive_reactions_count",
        "public_reactions_count",
        "comments_count",
        "reading_time_minutes",
        "language",
        "collection_id",
        "batch_id",
        "batch_ingestion_date",
        "has_comments",
        "has_reactions",
        "user_username",
        "user_name",
        "user_id",
        "date",
        "year",
        "month",
        "day"
    )
)
df_gold_articles_fact_table.limit(10).display()

# df_gold_full_articles_fact_table.write.format("delta") \
#     .partitionBy("year", "month") \
#     .mode("overwrite") \
#     .save(f"{gold_adsl_path}full_articles_fact")


In [0]:
# Not saved to gold, can be calculated in Power BI or similar tool from tags_fact
from pyspark.sql.functions import col, count, concat_ws, lit, avg, max, min, sum, round

df_articles_per_month_fact_table = (
    df_silver.groupBy("year", "month")
    .agg(
        count("id").alias("article_count"),
        avg("reading_time_minutes").alias("avg_reading_time"),
        round(sum(col("reading_time_minutes") / 60 / 24)).alias(
            "sum_reading_time_days"
        ),
    )
    .orderBy("year", "month")
)
# df_articles_per_month_fact_table.limit(10).display()

df_articles_per_month_fact_table.write.format("delta") \
    .mode("overwrite") \
    .save(f"{gold_adsl_path}articles_per_month_fact_table_no_partitions")

In [0]:
# Not saved to gold, can be calculated in Power BI or similar tool
from pyspark.sql.functions import count, sum

df_kpi = df_gold_articles_fact_table.agg(
    count("id").alias("total_articles"),
    sum("has_comments").alias("articles_with_comments"),
    sum("has_reactions").alias("articles_with_reactions"),
    (col("articles_with_comments") / col("total_articles") * 100).alias("comments_percentage"),
    (col("articles_with_reactions") / col("total_articles") * 100).alias("reactions_percentage"),
)
df_kpi.display()


total_articles,articles_with_comments,articles_with_reactions,comments_percentage,reactions_percentage
1038100,97637,264252,9.405355938734226,25.455351122242558


In [0]:
# Not saved to gold, can be calculated in Power BI or similar tool

from pyspark.sql.functions import col, when, avg, count, corr
from pyspark.sql import DataFrame

# Create reading time bins
df_binned = (
    df_silver.orderBy(col("positive_reactions_count").desc())
    .limit(10000)
    .withColumn(
        "reading_time_bin",
        when(col("reading_time_minutes") <= 5, "0-5")
        .when(
            (col("reading_time_minutes") > 5) & (col("reading_time_minutes") <= 10),
            "5-10",
        )
        .otherwise("10+"),
    )
)

# Aggregate engagement metrics per bin
df_bin_stats = (
    df_binned.groupBy("reading_time_bin")
    .agg(
        count("*").alias("articles_count"),
        avg("positive_reactions_count").alias("avg_reactions"),
        avg("comments_count").alias("avg_comments"),
    )
)
df_bin_stats.display()

# Example: correlation reading_time_minutes vs reactions within each bin
bins = ["0-5", "5-10", "10+"]

for b in bins:
    df_subset = df_binned.filter(col("reading_time_bin") == b)
    corr_val = df_subset.select(
        corr("reading_time_minutes", "positive_reactions_count")
    ).collect()[0][0]
    print(f"Bin {b} – correlation (reading time vs reactions): {corr_val:.3f}")

    corr_val_comments = df_subset.select(
        corr("reading_time_minutes", "comments_count")
    ).collect()[0][0]
    print(f"Bin {b} – correlation (reading time vs comments): {corr_val_comments:.3f}")

# Conclusion:
# Very short articles (<5min reading time) tend to have fewer comments
# There is no correlation between reading time and reactions or reading time and comments for articles longer than 5 minutes


reading_time_bin,articles_count,avg_reactions,avg_comments
0-5,6270,100.48341307814992,15.296969696969697
5-10,2639,118.00985221674877,11.900719969685486
10+,1091,142.6443629697525,13.146654445462879


Bin 0-5 – correlation (reading time vs reactions): 0.074
Bin 0-5 – correlation (reading time vs comments): -0.194
Bin 5-10 – correlation (reading time vs reactions): 0.056
Bin 5-10 – correlation (reading time vs comments): 0.039
Bin 10+ – correlation (reading time vs reactions): 0.106
Bin 10+ – correlation (reading time vs comments): 0.039


In [0]:
# Not saved to gold, can be calculated in Power BI or similar tool using the gold-tags-fact table
# Top tags
from pyspark.sql.functions import explode, col, count, row_number, desc

df_top_tags = df_silver.select(
    "reading_time_minutes",
    "comments_count",
    "public_reactions_count",
    explode(col("tag_list")).alias("tag"),
).filter(col("tag").isNotNull() & (col("tag") != "")) # filter articles with no tags

# filter top tags
df_top_tags = (
    df_top_tags.groupBy("tag")
    .agg(
        count("*").alias("articles_count"),
        avg("reading_time_minutes").alias("avg_reading_time"),
        avg("public_reactions_count").alias("avg_reactions"),
        avg("comments_count").alias("avg_comments"),
    )
    .filter(col("articles_count") > 1000) # filter rare tags
    .orderBy(col("avg_comments").desc())
    .limit(100)  # filter top tags
)

# Total number of unique tags
# total_unique_tags = df_top_tags_reading_time.count()
# Total unique tags: 120102
# print(f"Total unique tags: {total_unique_tags}")
df_top_tags.limit(100).display()

tag,articles_count,avg_reading_time,avg_reactions,avg_comments
watercooler,1324,2.4290030211480365,17.79380664652568,12.840634441087614
discuss,5500,2.641272727272727,15.231636363636364,9.320727272727273
codenewbie,1301,3.0953112990007687,17.19446579554189,6.082244427363566
career,3510,3.7384615384615385,21.84102564102564,5.66980056980057
devchallenge,2264,3.137809187279152,26.67226148409894,5.571554770318021
opensource,4652,4.36414445399828,36.59845227858985,5.527944969905417
css,2521,3.74851249504165,25.052360174533916,5.012296707655692
productivity,6042,3.98229063224098,24.656074147633234,4.876530950016551
github,1576,3.9143401015228427,28.679568527918782,4.691624365482234
coding,1487,4.1371889710827165,23.14593140551446,4.6476126429051785


In [0]:
from pyspark.sql.functions import col, lower, regexp_replace, split, explode, length
from pyspark.sql.functions import count

# Select and clean
df_titles = df_silver.select("id", "title", "year", "month") \
    .withColumn("title_clean",
                lower(regexp_replace(col("title"), "[^a-zA-Z0-9 ]", "")))
    
df_words = df_titles.withColumn(
    "word",
    explode(split(col("title_clean"), " "))
)

# Remove short words
df_words = df_words.filter(length(col("word")) > 1)
stopwords = ["this", "that", "with", "from", "your", "have", "will", "about", "using", "what", "to", "the", "in", "and", "for", "of", "how", "are", "an", "be", "is", "on", "as", "at", "by", "do", "if", "it", "no", "or", "so", "to", "up", "my", "we", "you", 'all', 'any', 'bed',]
df_words = df_words.filter(~col("word").isin(stopwords))

df_keywords = df_words.groupBy("word") \
    .agg(count("id").alias("article_count")) \
    .orderBy(col("article_count").desc())
# df_keywords.limit(100).display()

df_keywords_month = df_words.groupBy("year", "month", "word") \
    .agg(count("id").alias("article_count")) \
    .filter(col("article_count") > 200) \
    .orderBy("year", "month",)
df_keywords_month.display()



year,month,word,article_count
2022,12,web,275
2022,12,guide,209
2022,12,app,301
2022,12,data,304
2022,12,day,274
2022,12,create,219
2022,12,javascript,449
2022,12,2022,437
2022,12,best,220
2022,12,use,267


In [0]:
# Compare engagement between:
# Titles containing ? → Question
# Titles without ? → Non-question
from pyspark.sql.functions import col, when, avg, count

df_title_analysis = df_silver.withColumn(
    "is_question",
    when(col("title").contains("?"), "Question")
    .otherwise("Non-Question")
)

df_question_stats = df_title_analysis.groupBy("is_question") \
    .agg(
        count("id").alias("article_count"),
        avg("positive_reactions_count").alias("avg_reactions"),
        avg("comments_count").alias("avg_comments"),
        avg("reading_time_minutes").alias("avg_reading_time")
    )

df_question_stats.display()

is_question,article_count,avg_reactions,avg_comments,avg_reading_time
Question,75667,1.8332694569627446,0.6294553768485601,3.2605627287985515
Non-Question,962433,2.2777595946938645,0.34774472612639007,3.582382358044664


In [0]:
from pyspark.sql.functions import lower, regexp_replace, col, when, expr, split, avg, count, round

df_sent = df_silver.select("id", "title", "positive_reactions_count", "comments_count")

df_sent = df_sent.withColumn(
    "title_clean",
    lower(regexp_replace(col("title"), "[^a-zA-Z0-9 ]", ""))
)



df_sent = df_sent.withColumn("words", split(col("title_clean"), " "))
positive_words = ["best", "great", "amazing", "awesome", "fast", "easy", "powerful", "improve"]
negative_words = ["bad", "slow", "worst", "hard", "difficult", "problem", "issue", "error"]



df_sent = df_sent.withColumn(
    "positive_hits",
    expr(f"size(array_intersect(words, array({','.join([f'\"{w}\"' for w in positive_words])})))")
)
df_sent = df_sent.withColumn(
    "negative_hits",
    expr(f"size(array_intersect(words, array({','.join([f'\"{w}\"' for w in negative_words])})))")
)
df_sent = df_sent.withColumn(
    "sentiment_score",
    col("positive_hits") - col("negative_hits")
)


df_sent = df_sent.withColumn(
    "sentiment_category",
    when(col("sentiment_score") > 0, "Positive")
    .when(col("sentiment_score") < 0, "Negative")
    .otherwise("Neutral")
)


df_sentiment_stats = df_sent.groupBy("sentiment_category") \
    .agg(
        count("id").alias("article_count"),
        round(avg("positive_reactions_count"), 2).alias("avg_reactions"),
        round(avg("comments_count"), 2).alias("avg_comments")
    )

df_sentiment_stats.display()

sentiment_category,article_count,avg_reactions,avg_comments
Neutral,979032,2.24,0.37
Positive,49539,2.33,0.3
Negative,9529,2.36,0.52


In [0]:
# Title Patterns
# Numbers in titles ("10 ways to...")
# "How to" titles
# "Beginner guide"

# Numbers in titles:
from pyspark.sql.functions import col, regexp_replace, length, when, expr, split, avg, count, round

conditionComments = (col("comments_count") > 0) & (col("comments_count").isNotNull())
conditionPublicReactions = (col("public_reactions_count") > 0) & (
    col("public_reactions_count").isNotNull()
)

# Articles with comments or reactions
df_silver_with_rections_title = df_silver.filter(conditionComments | conditionPublicReactions)

df_silver_with_rections_title_patterns = (df_silver_with_rections_title.withColumn("title_clean",
                lower(regexp_replace(col("title"), "[^a-zA-Z0-9 ]", ""))))

df_silver_with_rections_title_patterns_all = (df_silver.withColumn("title_clean",
                lower(regexp_replace(col("title"), "[^a-zA-Z0-9 ]", ""))))
# number_list = [5, 7, 8, 10, 11, 12, 15, 18, 20, 25, 27, 30]
# Create a regex pattern from number_list with word boundaries
# number_pattern = r"\b(" + "|".join(map(str, number_list)) + r")\b"

keyword = 'performance'

df_title_analysis_keyword = (df_silver_with_rections_title_patterns.withColumn(
    f"is_{keyword}",
    when(col("title_clean").contains(keyword), f"{keyword}")
    .otherwise(f"Non-{keyword}")
))
df_keyword_stats = df_title_analysis_keyword.groupBy(f"is_{keyword}") \
    .agg(
        count("id").alias("article_count"),
        avg("positive_reactions_count").alias("avg_reactions"),
        avg("comments_count").alias("avg_comments"),
        avg("reading_time_minutes").alias("avg_reading_time")
    )
df_keyword_stats.display()

df_title_analysis_keyword_all = (df_silver_with_rections_title_patterns_all.withColumn(
    f"is_{keyword}",
    when(col("title_clean").contains(keyword), f"{keyword}")
    .otherwise(f"Non-{keyword}")
))
df_keyword_stats_all = df_title_analysis_keyword_all.groupBy(f"is_{keyword}") \
    .agg(
        count("id").alias("article_count"),
        avg("positive_reactions_count").alias("avg_reactions"),
        avg("comments_count").alias("avg_comments"),
        avg("reading_time_minutes").alias("avg_reading_time")
    )
df_keyword_stats_all.display()



is_performance,article_count,avg_reactions,avg_comments,avg_reading_time
Non-performance,282136,8.193704454589277,1.3461522102815664,4.039459693197607
performance,3064,6.754242819843342,0.825065274151436,4.927219321148825


is_performance,article_count,avg_reactions,avg_comments,avg_reading_time
Non-performance,1028434,2.2463463868366857,0.3692818401569765,3.5475859413438298
performance,9666,2.1404924477550176,0.26153527829505485,4.76536312849162
